# 05 — Sample-size curves (pooled + LOCO)

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.lib.cv import (sample_size_curve, sample_size_curve_loco,
                             sample_size_curve_baselines, loco_cv)
from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.plotting import VARIABLE_LABELS, PALETTE

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
OUT_DIR    = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "sample_size_curves"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CURVES_PATH      = OUT_DIR / "sample_size_curves.csv"
CURVES_LOCO_PATH = OUT_DIR / "sample_size_curves_loco.csv"

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

FRACTIONS = (0.1, 0.25, 0.5, 0.75, 1.0)
N_OUTER_FOLDS = 10
N_BOOTSTRAPS = 10
SEED = 42
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_LOCO = False

In [ ]:
# Pooled compute (cached)
if not FORCE_RECOMPUTE and CURVES_PATH.exists():
    print(f"Cached results at {CURVES_PATH} — skipping compute.")
else:
    clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
    acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
    clinical = clinical_df.loc[acute_pool]
    X_demo, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
    lesion = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
    flnm   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
    slnm   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)
    targets = identify_binary_targets(clinical)

    all_curves = []
    for tgt in targets:
        valid = clinical[tgt].notna()
        y = clinical.loc[valid, tgt].values
        X_base = X_demo[valid.values]
        n_demo = X_base.shape[1]
        arm_a = [("baseline", X_base)]
        arm_b = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, lesion.loc[valid].values])),
            ("flnm_acute",   np.hstack([X_base, flnm.loc[valid].values])),
            ("slnm_acute",   np.hstack([X_base, slnm.loc[valid].values])),
        ]
        print(f"--- {tgt} (N={len(y)}) ---")
        df = sample_size_curve(arm_a, arm_b, y, n_demo, fractions=FRACTIONS,
                               n_outer_folds=N_OUTER_FOLDS,
                               n_bootstraps=N_BOOTSTRAPS, seed=SEED)
        df["Target"] = tgt
        all_curves.append(df)
    out = pd.concat(all_curves, ignore_index=True)
    out.to_csv(CURVES_PATH, index=False)
    print(f"Saved {len(out)} rows to {CURVES_PATH}")

In [ ]:
# Append Majority + Permuted reference arms (idempotent)
existing = pd.read_csv(CURVES_PATH)
have_pairs = set(zip(existing["Target"], existing["arm"]))
arms_wanted = ["majority", "permuted"]

clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]
X_demo, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
targets = identify_binary_targets(clinical)

new_rows = []
for tgt in targets:
    missing_arms = [a for a in arms_wanted if (tgt, a) not in have_pairs]
    if not missing_arms:
        continue
    valid = clinical[tgt].notna()
    y = clinical.loc[valid, tgt].values
    X_base = X_demo[valid.values]
    df = sample_size_curve_baselines(y, X_base, fractions=FRACTIONS,
                                     n_outer_folds=N_OUTER_FOLDS,
                                     n_bootstraps=N_BOOTSTRAPS, seed=SEED)
    df = df[df["arm"].isin(missing_arms)].copy()
    df["Target"] = tgt
    new_rows.append(df)
    print(f"  {tgt}: appended arms {sorted(df['arm'].unique())}")
if new_rows:
    existing = pd.concat([existing] + new_rows, ignore_index=True)
    existing.to_csv(CURVES_PATH, index=False)
curves = pd.read_csv(CURVES_PATH)
print(f"{len(curves)} rows loaded")

In [ ]:
# Pooled plot: 7-panel grid, x = actual training-set size
sns.set_theme(style="whitegrid", font_scale=1.2)

ARM_LABELS = {
    "majority": "Majority (test prevalence)",
    "permuted": "Permuted (shuffled y_train)",
    "a":        "Baseline (clinical-only)",
    "b":        "Stacker (baseline + Lesion + fLNM + sLNM)",
}
ARM_COLORS = {"majority": PALETTE["Majority"], "permuted": PALETTE["Permuted"],
              "a": PALETTE["Baseline"], "b": PALETTE["Stacking"]}
ARM_LINESTYLE = {"majority": ":", "permuted": "--", "a": "-", "b": "-"}

target_n = {t: int(clinical_df[t].notna().sum()) for t in VARIABLE_LABELS
            if t in clinical_df.columns}
TRAIN_FRACTION = (N_OUTER_FOLDS - 1) / N_OUTER_FOLDS

targets_order = [t for t in VARIABLE_LABELS if t in curves["Target"].unique()]
n_cols = 4
n_rows = int(np.ceil(len(targets_order) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.0 * n_rows), sharey=True)
axes = np.atleast_1d(axes).ravel()

for ax, tgt in zip(axes, targets_order):
    sub = curves[curves["Target"] == tgt]
    full_n = target_n.get(tgt, np.nan)
    train_n = full_n * TRAIN_FRACTION
    for arm, color in ARM_COLORS.items():
        a_sub = sub[sub["arm"] == arm]
        if a_sub.empty:
            continue
        # Two-stage SE: average within fold first, then SEM across folds
        fold_means = a_sub.groupby(["frac", "fold"])["AUPRC"].mean().reset_index()
        agg = fold_means.groupby("frac")["AUPRC"].agg(["mean", "sem"]).reset_index()
        agg["n"] = (agg["frac"] * train_n).round().astype(int)
        is_headline = arm in ("a", "b")
        ax.plot(agg["n"], agg["mean"], marker="o" if is_headline else None,
                linestyle=ARM_LINESTYLE[arm], color=color, markersize=5,
                linewidth=1.6 if is_headline else 2.2,
                label=ARM_LABELS[arm], zorder=4 if is_headline else 5)
        if is_headline:
            ax.fill_between(agg["n"], agg["mean"] - agg["sem"], agg["mean"] + agg["sem"],
                            color=color, alpha=0.22, zorder=2)
    ax.set_title(f"{VARIABLE_LABELS[tgt]}  (N={int(full_n)})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Training-set size (n subjects)")
    ax.set_ylabel("AUPRC")
    xticks = [int(round(f * train_n)) for f in sorted(curves["frac"].unique())]
    ax.set_xticks(xticks)
    ax.grid(alpha=0.3)

for ax in axes[len(targets_order):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=True,
           bbox_to_anchor=(0.5, -0.02), fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
fig.savefig(FIG_DIR / "sample_size_curves.png", dpi=200, bbox_inches="tight")
print(f"saved {FIG_DIR / 'sample_size_curves.png'}")

In [ ]:
# LOCO compute (cached)
if not FORCE_RECOMPUTE_LOCO and CURVES_LOCO_PATH.exists():
    print(f"Cached LOCO results at {CURVES_LOCO_PATH} — skipping compute.")
else:
    X_demo_l, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=True)
    lesion = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
    flnm   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
    slnm   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)
    loco_splits = list(loco_cv(clinical["demographics-cohort"]))

    all_curves = []
    for tgt in targets:
        valid = clinical[tgt].notna()
        y = clinical.loc[valid, tgt].values
        X_base = X_demo_l[valid.values]
        n_demo = X_base.shape[1]
        arm_a = [("baseline", X_base)]
        arm_b = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, lesion.loc[valid].values])),
            ("flnm_acute",   np.hstack([X_base, flnm.loc[valid].values])),
            ("slnm_acute",   np.hstack([X_base, slnm.loc[valid].values])),
        ]
        valid_idx = np.where(valid.values)[0]
        mapping = {orig: i for i, orig in enumerate(valid_idx)}
        target_splits = []
        for train, test in loco_splits:
            tr = np.array([mapping[o] for o in train if o in mapping], dtype=int)
            te = np.array([mapping[o] for o in test if o in mapping], dtype=int)
            if len(te) > 0 and len(tr) > 0 and len(np.unique(y[te])) == 2:
                target_splits.append((tr, te))
        if not target_splits:
            print(f"--- {tgt}: no LOCO folds with both classes — skipped")
            continue
        print(f"--- {tgt} (N={len(y)}, {len(target_splits)} LOCO folds) ---")
        df = sample_size_curve_loco(arm_a, arm_b, y, target_splits, n_demo,
                                    fractions=FRACTIONS,
                                    n_bootstraps=N_BOOTSTRAPS, seed=SEED)
        df["Target"] = tgt
        all_curves.append(df)
    out_loco = pd.concat(all_curves, ignore_index=True)
    out_loco.to_csv(CURVES_LOCO_PATH, index=False)
    print(f"Saved {len(out_loco)} rows to {CURVES_LOCO_PATH}")

In [ ]:
# Append Majority + Permuted arms to the LOCO CSV (idempotent)
existing = pd.read_csv(CURVES_LOCO_PATH)
have_pairs = set(zip(existing["Target"], existing["arm"]))
X_demo_l, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=True)
loco_splits = list(loco_cv(clinical["demographics-cohort"]))

new_rows = []
for tgt in targets:
    missing_arms = [a for a in arms_wanted if (tgt, a) not in have_pairs]
    if not missing_arms:
        continue
    valid = clinical[tgt].notna()
    y = clinical.loc[valid, tgt].values
    X_base = X_demo_l[valid.values]
    valid_idx = np.where(valid.values)[0]
    mapping = {orig: i for i, orig in enumerate(valid_idx)}
    target_splits = []
    for train, test in loco_splits:
        tr = np.array([mapping[o] for o in train if o in mapping], dtype=int)
        te = np.array([mapping[o] for o in test if o in mapping], dtype=int)
        if len(te) > 0 and len(tr) > 0 and len(np.unique(y[te])) == 2:
            target_splits.append((tr, te))
    if not target_splits:
        continue
    df = sample_size_curve_baselines(y, X_base, fractions=FRACTIONS,
                                     n_bootstraps=N_BOOTSTRAPS, seed=SEED,
                                     splits=target_splits)
    df = df[df["arm"].isin(missing_arms)].copy()
    df["Target"] = tgt
    new_rows.append(df)
if new_rows:
    existing = pd.concat([existing] + new_rows, ignore_index=True)
    existing.to_csv(CURVES_LOCO_PATH, index=False)
curves_loco = pd.read_csv(CURVES_LOCO_PATH)
print(f"{len(curves_loco)} LOCO rows loaded")

In [ ]:
# LOCO plot: x = actual mean training-set size (n_train column)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.0 * n_rows), sharey=True)
axes = np.atleast_1d(axes).ravel()
targets_order_l = [t for t in VARIABLE_LABELS if t in curves_loco["Target"].unique()]

for ax, tgt in zip(axes, targets_order_l):
    sub = curves_loco[curves_loco["Target"] == tgt]
    full_n = target_n.get(tgt, np.nan)
    for arm, color in ARM_COLORS.items():
        a_sub = sub[sub["arm"] == arm]
        if a_sub.empty:
            continue
        fold_means = a_sub.groupby(["frac", "fold"]).agg(
            AUPRC=("AUPRC", "mean"), n_train=("n_train", "mean")).reset_index()
        agg = fold_means.groupby("frac").agg(
            mean=("AUPRC", "mean"), sem=("AUPRC", "sem"),
            n_train=("n_train", "mean")).reset_index()
        is_headline = arm in ("a", "b")
        ax.plot(agg["n_train"], agg["mean"], marker="o" if is_headline else None,
                linestyle=ARM_LINESTYLE[arm], color=color, markersize=5,
                linewidth=1.6 if is_headline else 2.2,
                label=ARM_LABELS[arm], zorder=4 if is_headline else 5)
        if is_headline:
            ax.fill_between(agg["n_train"], agg["mean"] - agg["sem"],
                            agg["mean"] + agg["sem"], color=color, alpha=0.22, zorder=2)
    ax.set_title(f"{VARIABLE_LABELS[tgt]}  (N={int(full_n)})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Mean training-set size (n subjects)")
    ax.set_ylabel("AUPRC")
    tick_sub = sub[sub["arm"] == "b"]
    if not tick_sub.empty:
        xticks = (tick_sub.groupby("frac")["n_train"].mean()
                  .sort_values().round().astype(int).tolist())
        ax.set_xticks(xticks)
    ax.grid(alpha=0.3)

for ax in axes[len(targets_order_l):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=True,
           bbox_to_anchor=(0.5, -0.02), fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
fig.savefig(FIG_DIR / "sample_size_curves_loco.png", dpi=200, bbox_inches="tight")
print(f"saved {FIG_DIR / 'sample_size_curves_loco.png'}")